In [70]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.io as pio

import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pointbiserialr

pio.templates.default = "plotly_dark"

In [71]:
df_healthy = pd.read_parquet("../../data/processed/dataset_features.parquet")


In [72]:
pd.set_option("display.max_seq_items", None)
df_healthy.columns

Index(['degree_AF3', 'clustering_AF3', 'betweenness_AF3',
       'local_efficiency_AF3', 'hub_frequency_AF3', 'weighted_degree_AF3',
       'degree_AF4', 'clustering_AF4', 'betweenness_AF4',
       'local_efficiency_AF4', 'hub_frequency_AF4', 'weighted_degree_AF4',
       'degree_AF8', 'clustering_AF8', 'betweenness_AF8',
       'local_efficiency_AF8', 'hub_frequency_AF8', 'weighted_degree_AF8',
       'degree_AFz', 'clustering_AFz', 'betweenness_AFz',
       'local_efficiency_AFz', 'hub_frequency_AFz', 'weighted_degree_AFz',
       'degree_C1', 'clustering_C1', 'betweenness_C1', 'local_efficiency_C1',
       'hub_frequency_C1', 'weighted_degree_C1', 'degree_C2', 'clustering_C2',
       'betweenness_C2', 'local_efficiency_C2', 'hub_frequency_C2',
       'weighted_degree_C2', 'degree_C3', 'clustering_C3', 'betweenness_C3',
       'local_efficiency_C3', 'hub_frequency_C3', 'weighted_degree_C3',
       'degree_C4', 'clustering_C4', 'betweenness_C4', 'local_efficiency_C4',
       'hub_freq

In [73]:

print(df_healthy["age"].value_counts())
print(df_healthy["gender"].value_counts())
print(df_healthy["handedness"].value_counts())
print(df_healthy["education"].value_counts())
print(df_healthy["drug"].value_counts())

# garantir tipos
df_healthy["age"] = df_healthy["age"].astype(str)
df_healthy["gender"] = df_healthy["gender"].astype(str)
df_healthy["handedness"] = df_healthy["handedness"].astype(str)
df_healthy["education"] = df_healthy["education"].astype(str)
df_healthy["drug"] = df_healthy["drug"].astype(str)


age
20-25    36
25-30    28
60-65    12
70-75    12
65-70    10
30-35     6
55-60     2
75-80     2
Name: count, dtype: int64
gender
2    84
1    24
Name: count, dtype: int64
handedness
right          96
left           10
ambedextor      2
Name: count, dtype: int64
education
gymnasium     80
realschule    26
gymansium      2
Name: count, dtype: int64
drug
0.0    108
Name: count, dtype: int64


In [74]:
# Organizando as variáveis (dicotomia alinhada ao LEMON: young ~20–35, older ~59–77+)
df_healthy["age_group"] = df_healthy["age"].replace({
    "20-25": "young",
    "25-30": "young",
    "30-35": "young",
    "35-40": "young",
    "55-60": "older",
    "60-65": "older",
    "65-70": "older",
    "70-75": "older",
    "75-80": "older",
})

# Calculando as médias das métricas
degree_cols = [c for c in df_healthy.columns if c.startswith("degree_")]
clustering_cols = [c for c in df_healthy.columns if c.startswith("clustering_")]
betweenness_cols = [c for c in df_healthy.columns if c.startswith("betweenness_")]
sd_degree_cols = [c for c in df_healthy.columns if c.startswith("sd_degree_")]
bin_clustering_cols = [c for c in df_healthy.columns if c.startswith("bin_clustering_")]
df_healthy["degree_mean"] = df_healthy[degree_cols].mean(axis=1)
df_healthy["clustering_mean"] = df_healthy[clustering_cols].mean(axis=1)
df_healthy["betweenness_mean"] = df_healthy[betweenness_cols].mean(axis=1)
df_healthy["sd_degree_mean"] = df_healthy[sd_degree_cols].mean(axis=1)
df_healthy["bin_clustering_mean"] = df_healthy[bin_clustering_cols].mean(axis=1)


In [75]:
df_healthy["age"].value_counts()

age
20-25    36
25-30    28
60-65    12
70-75    12
65-70    10
30-35     6
55-60     2
75-80     2
Name: count, dtype: int64

In [76]:

GRAPH_METRICS = {
    "degree_mean": "Grau médio por janela (normalizado)",
    "sd_degree_mean": "Variabilidade temporal do grau (DP)",
    "bin_clustering_mean": "Agrupamento binário por janela",
    "clustering_mean": "Agrupamento ponderado (REA)",
    "betweenness_mean": "Intermediação (REA, distância 1/peso)",
}

# a célula seguinte renomeia age_group/gender; aceita os dois nomes para permitir reexecução
age_col = "age_group" if "age_group" in df_healthy.columns else "Faixa Etária"
gender_col = "gender" if "gender" in df_healthy.columns else "Gênero"

sample_summary = (
    df_healthy[["subject_id", age_col, gender_col]]
    .drop_duplicates()
    .groupby([age_col, gender_col])
    .size()
    .reset_index(name="n")
)

sample_summary

,age_group,gender,n
0,older,1,4
1,older,2,15
2,young,1,8
3,young,2,27


In [77]:
df_healthy = (
    df_healthy[df_healthy["condition"] != "E0"]
    .rename(columns={"age_group": "Faixa Etária", "gender": "Gênero"})
    .copy()
)

df_healthy[["Faixa Etária", "Gênero"]].value_counts()

Faixa Etária  Gênero
young         2         54
older         2         30
young         1         16
older         1          8
Name: count, dtype: int64

### Idade

In [78]:
for metric, label in GRAPH_METRICS.items():
    fig = px.box(
        df_healthy,
        x="Faixa Etária",
        y=metric,
        color="Faixa Etária",
        title=f"{label} vs Age Group",
        points='all',
        labels={
            "Faixa Etária": "Faixa Etária",
            metric: label
        }
    )
    fig.update_layout(
        xaxis_title="Faixa Etária",
        yaxis_title=label,
        title=f"{label} vs Faixa Etária"
    )
    fig.show()

-------

### Condition

In [79]:

# plotly
for metric, label in GRAPH_METRICS.items():
    fig = px.box(
        df_healthy,
        x="condition",
        y=metric,
        color="condition",
        title=f"{metric} vs Condition",
        points="all",
        labels={
            "condition": "Condition",
            metric: metric
        }
    )
    fig.update_layout(
        xaxis_title="Condition",
        yaxis_title=label,
        title=f"{label} vs Condition"
    )
    fig.show()

------

In [80]:
# Boxplot Age Group x Condition
for metric, label in GRAPH_METRICS.items():
    fig = px.box(
        df_healthy,
        x="Faixa Etária",
        y=metric,
        color="condition",
        title=f"{label} vs Faixa Etária",
        points="all",
        labels={
            "Faixa Etária": "Faixa Etária",
            label: label
        }
    )
    fig.update_layout(
        xaxis_title=f"{label}: EO vs EC estratificado por idade",
        yaxis_title=label,
        title=f"{label} vs Faixa Etária"
    )
    fig.show()

### Figura 6 complementar — idade × EO/EC (painéis A–C)

Mesmas métricas da Figura 6, agora com **EO vs EC** lado a lado em cada faixa etária. Layout vertical (A/B/C) para legibilidade na página, com **uma legenda** comum.

In [81]:
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots

PANEL_METRICS = [
    ("degree_mean", "Grau médio por janela (normalizado)", "A"),
    ("bin_clustering_mean", "Agrupamento binário por janela", "B"),
    ("betweenness_mean", "Intermediação (REA)", "C"),
]

age_order = ["young", "older"]
cond_order = ["EO", "EC"]
color_map = {"EO": "#636EFA", "EC": "#EF553B"}

fig6 = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=[f"{tag}. {lab}" for _, lab, tag in PANEL_METRICS],
)

for row, (metric, ylabel, _tag) in enumerate(PANEL_METRICS, start=1):
    for cond in cond_order:
        sub = df_healthy.loc[df_healthy["condition"] == cond]
        fig6.add_trace(
            go.Box(
                x=sub["Faixa Etária"],
                y=sub[metric],
                name=cond,
                legendgroup=cond,
                marker_color=color_map[cond],
                line_color=color_map[cond],
                offsetgroup=cond,
                boxpoints="all",
                jitter=0.25,
                pointpos=0,
                showlegend=(row == 1),
            ),
            row=row,
            col=1,
        )
    fig6.update_yaxes(title_text=ylabel, row=row, col=1)
    fig6.update_xaxes(
        categoryorder="array",
        categoryarray=age_order,
        row=row,
        col=1,
    )

fig6.update_xaxes(title_text="Faixa etária", row=3, col=1)
fig6.update_layout(
    height=900,
    width=700,
    boxmode="group",
    legend_title_text="Condição",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(t=90, b=60),
    title_text="Figura 6 complementar: métricas de rede por faixa etária e condição (EO/EC)",
)
fig6.for_each_annotation(lambda a: a.update(font_size=12))
fig6.show()

out_dir = Path("../../figures")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "fig6_age_eo_ec_panels.png"
try:
    fig6.write_image(str(out_path), scale=2)
    print(f"Salvo: {out_path.resolve()}")
except Exception as e:
    html_path = out_dir / "fig6_age_eo_ec_panels.html"
    fig6.write_html(str(html_path))
    print(f"PNG não exportado ({type(e).__name__}: {e}). HTML: {html_path}")


PNG não exportado (ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
). HTML: ..\..\figures\fig6_age_eo_ec_panels.html


------------------

### Correlações

In [82]:
# Idade com um valor por participante: EO e EC do mesmo sujeito não são independentes
from scipy.stats import false_discovery_control, mannwhitneyu

per_subject = df_healthy.pivot_table(
    index=["subject_id", "Faixa Etária"], columns="condition", values=list(GRAPH_METRICS)
)


def age_test(values: pd.Series) -> dict:
    values = values.dropna()
    older = values.xs("older", level="Faixa Etária")
    young = values.xs("young", level="Faixa Etária")
    u, p = mannwhitneyu(older, young, alternative="two-sided")
    return {
        "n_older": len(older),
        "n_young": len(young),
        "mediana_older": older.median(),
        "mediana_young": young.median(),
        "p_value": p,
        # negativo = menor nos idosos
        "rank_biserial": 2 * u / (len(older) * len(young)) - 1,
    }


age_rows, interaction_rows = [], []
for metric, label in GRAPH_METRICS.items():
    eo, ec = per_subject[(metric, "EO")], per_subject[(metric, "EC")]
    age_rows.append({"metric": label, "medida": "média EO/EC", **age_test((eo + ec) / 2)})
    age_rows.append({"metric": label, "medida": "só EO (descritivo)", **age_test(eo)})
    age_rows.append({"metric": label, "medida": "só EC (descritivo)", **age_test(ec)})
    interaction_rows.append({"metric": label, "medida": "EO − EC", **age_test(eo - ec)})

age_tests = pd.DataFrame(age_rows)
main = age_tests["medida"] == "média EO/EC"
age_tests.loc[main, "p_fdr"] = false_discovery_control(age_tests.loc[main, "p_value"])
age_tests.round(4)

,metric,spearman_r,p_value
0,degree_mean,-0.425403,0.000004
1,clustering_mean,-0.422915,0.000005
2,betweenness_mean,-0.253127,0.008214


Com um valor por participante (35 jovens, 19 idosos; média de EO e EC), os idosos apresentaram, após correção FDR:

- **menor grau médio por janela** (mediana 0,51 vs 0,58; p = 0,001; rank-biserial = −0,55): em cada janela, os eletrodos se sincronizam com uma fração menor dos demais;
- **menor agrupamento binário por janela** (0,69 vs 0,74; p < 0,001; r = −0,59) e menor agrupamento ponderado da REA (p = 0,002; r = −0,53);
- **menor variabilidade temporal do grau** (DP 0,178 vs 0,183; p = 0,012; r = −0,42).

A intermediação tende a ser maior nos idosos, mas não foi significativa na média EO/EC (p = 0,073); a diferença aparece em EO (p = 0,011, descritivo).

Uma interpretação para esses resultados é que, com o envelhecimento, as redes funcionais de cada instante (janelas do TVG) ficam mais esparsas e menos agrupadas: cada eletrodo se sincroniza com menos eletrodos por vez, e seus vizinhos formam menos triângulos entre si. A esparsificação vale para as redes por janela; a rede agregada (REA) é completa em todos os participantes e não muda de densidade. A menor variabilidade do grau indica que essa sincronização, além de menor, oscila menos no tempo. A tendência de maior intermediação nos idosos, mais clara em EO, sugere que, com menos ligações diretas fortes, alguns eletrodos passam a funcionar como ponte. Esses achados vêm de uma amostra pequena e desbalanceada (19 idosos, poucos do sexo feminino) e devem ser lidos como exploratórios.

### Métricas por janela: todos os pontos

Cada ponto é um eletrodo numa janela do TVG (1967 janelas × ~60 eletrodos por registro). Esses pontos servem para ver a distribuição e a variabilidade no tempo, mas não para testes: janelas consecutivas de 100 ms são muito correlacionadas e vêm da mesma pessoa. Os testes acima usam os resumos por participante (média e desvio padrão no tempo).

- **Nuvem:** amostra de 200 pontos por registro, para o gráfico continuar leve.
- **Densidade:** histograma de todos os pontos de cada registro, normalizado e depois tirada a média dentro do grupo (cada registro pesa igual).

In [ ]:
from pathlib import Path

WINDOWS_DIR = Path("../../data/intermediate/windows")
POINTS_PER_RECORDING = 200
WINDOW_METRICS = {
    "degree": "Grau por janela (normalizado)",
    "clustering": "Agrupamento binário por janela",
}
# grau por janela é discreto (k / (N − 1), N ≈ 56–61): bins largos evitam serrilhado
bins = np.linspace(0, 1, 21)
rng = np.random.default_rng(42)

recordings = (
    df_healthy[["subject_id", "condition", "Faixa Etária"]]
    .drop_duplicates()
    .rename(columns={"Faixa Etária": "age_group"})
)

cloud_parts, hist_parts = [], []
for rec in recordings.itertuples(index=False):
    candidates = [WINDOWS_DIR / f"{rec.subject_id}_{c}.npz" for c in (rec.condition, "E0")]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        continue
    with np.load(path) as w:
        n_nodes = w["degree"].shape[0]
        series = {"degree": w["degree"] / (n_nodes - 1), "clustering": w["clustering"]}
    for key, label in WINDOW_METRICS.items():
        values = series[key].ravel()
        values = values[np.isfinite(values)]
        counts, _ = np.histogram(values, bins=bins)
        hist_parts.append(pd.DataFrame({
            "metric": label, "condition": rec.condition, "age_group": rec.age_group,
            "subject_id": rec.subject_id, "x": (bins[:-1] + bins[1:]) / 2,
            "densidade": counts / counts.sum() / np.diff(bins),
        }))
        cloud_parts.append(pd.DataFrame({
            "metric": label, "condition": rec.condition, "age_group": rec.age_group,
            "valor": rng.choice(values, size=min(POINTS_PER_RECORDING, len(values)), replace=False),
        }))

window_cloud = pd.concat(cloud_parts, ignore_index=True)
window_hist = pd.concat(hist_parts, ignore_index=True)
print(f"Registros com séries por janela: {window_hist['subject_id'].nunique()} sujeitos, "
      f"{len(hist_parts) // len(WINDOW_METRICS)} registros")

In [ ]:
fig = px.violin(
    window_cloud,
    x="age_group",
    y="valor",
    color="condition",
    facet_col="metric",
    points="all",
    box=True,
    category_orders={
        "age_group": ["young", "older"],
        "condition": ["EO", "EC"],
        "metric": list(WINDOW_METRICS.values()),
    },
    labels={"age_group": "Faixa etária", "valor": "Valor na janela", "condition": "Condição"},
    title="Métricas por janela: eletrodo × janela (amostra por registro)",
)
fig.update_traces(marker=dict(size=2, opacity=0.25), jitter=0.4, pointpos=0, spanmode="hard")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_yaxes(matches=None, showticklabels=True)
fig.show()

In [ ]:
group_density = (
    window_hist.groupby(["metric", "condition", "age_group", "x"], as_index=False)["densidade"].mean()
)

fig = px.line(
    group_density,
    x="x",
    y="densidade",
    color="age_group",
    line_dash="condition",
    facet_col="metric",
    category_orders={
        "age_group": ["young", "older"],
        "condition": ["EO", "EC"],
        "metric": list(WINDOW_METRICS.values()),
    },
    labels={"x": "Valor na janela", "densidade": "Densidade", "age_group": "Faixa etária", "condition": "Condição"},
    title="Distribuição das métricas por janela (média das densidades dos registros)",
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(matches=None)
fig.show()

Gênero

In [83]:
df_healthy["Gênero"].value_counts()

Gênero
2    84
1    24
Name: count, dtype: int64

In [84]:
df_healthy["Gênero"].dtype
df_healthy["Gênero"].unique()


array(['2', '1'], dtype=object)

In [85]:
df_healthy["Gênero"] = df_healthy["Gênero"].astype(float).astype(int)

df_healthy["gender_bin"] = df_healthy["Gênero"].map({
    1: 0,  # female
    2: 1   # male
})

df_healthy["gender_bin"].value_counts()


gender_bin
1    84
0    24
Name: count, dtype: int64

In [86]:
gender_results = []

for metric in GRAPH_METRICS:
    valid = df_healthy[[metric, "gender_bin"]].dropna()
    r, p = pointbiserialr(valid["gender_bin"], valid[metric])
    gender_results.append({
        "metric": metric,
        "point_biserial_r": r,
        "p_value": p
    })

pd.DataFrame(gender_results)



,metric,point_biserial_r,p_value
0,degree_mean,-0.053790,0.580331
1,clustering_mean,-0.099166,0.307212
2,betweenness_mean,0.039230,0.686872


### Comparação por gênero das métricas de rede

Codificação do metadado: `1 = feminino`, `2 = masculino`.

EO e EC vêm do mesmo sujeito, então os testes são feitos **separadamente por condição** (uma linha por sujeito), com Mann-Whitney bilateral e correção de Benjamini–Hochberg sobre os 10 testes (5 métricas × 2 condições). O tamanho de efeito é a correlação rank-biserial: valores positivos indicam métrica maior no grupo feminino.

O grupo feminino é pequeno, então a comparação tem baixo poder estatístico.

In [92]:
gender_label_map = {1: "Feminino", 2: "Masculino"}
gender_order = ["Feminino", "Masculino"]

df_healthy["Gênero_label"] = (
    pd.to_numeric(df_healthy["Gênero"]).astype(int).map(gender_label_map)
)

# sujeitos por gênero e faixa etária (checa se idade confunde a comparação)
(
    df_healthy.drop_duplicates("subject_id")
    .groupby(["Faixa Etária", "Gênero_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=gender_order)
)

Gênero_label,Feminino,Masculino
Faixa Etária,,
older,4,15
young,8,27


In [93]:
for metric, label in GRAPH_METRICS.items():
    fig = px.box(
        df_healthy,
        x="Gênero_label",
        y=metric,
        color="Gênero_label",
        points="all",
        category_orders={"Gênero_label": gender_order},
        labels={"Gênero_label": "Gênero", metric: label},
    )
    fig.update_layout(
        xaxis_title="Gênero",
        yaxis_title=label,
        title=f"{label} vs Gênero"
    )
    fig.show()

In [94]:
# Boxplot Gênero x Condition
for metric, label in GRAPH_METRICS.items():
    fig = px.box(
        df_healthy,
        x="Gênero_label",
        y=metric,
        color="condition",
        points="all",
        category_orders={"Gênero_label": gender_order, "condition": ["EO", "EC"]},
        labels={"Gênero_label": "Gênero", "condition": "Condição", metric: label},
    )
    fig.update_layout(
        xaxis_title="Gênero",
        yaxis_title=label,
        title=f"{label} vs Gênero (EO e EC)"
    )
    fig.show()

In [95]:
# Boxplot Gênero x Faixa Etária, separado por condição
for metric, label in GRAPH_METRICS.items():
    fig = px.box(
        df_healthy,
        x="Faixa Etária",
        y=metric,
        color="Gênero_label",
        facet_col="condition",
        points="all",
        category_orders={
            "Faixa Etária": ["young", "older"],
            "Gênero_label": gender_order,
            "condition": ["EO", "EC"],
        },
        labels={"Gênero_label": "Gênero", "condition": "Condição", metric: label},
    )
    fig.update_layout(title=f"{label} vs Gênero por Faixa Etária")
    fig.show()

In [96]:
from scipy.stats import false_discovery_control, mannwhitneyu

gender_mw_rows = []
for cond in ["EO", "EC"]:
    sub = df_healthy[df_healthy["condition"] == cond]
    for metric, label in GRAPH_METRICS.items():
        fem = sub.loc[sub["Gênero_label"] == "Feminino", metric].dropna()
        masc = sub.loc[sub["Gênero_label"] == "Masculino", metric].dropna()
        u, p = mannwhitneyu(fem, masc, alternative="two-sided")
        gender_mw_rows.append({
            "condition": cond,
            "metric": label,
            "n_fem": len(fem),
            "n_masc": len(masc),
            "mediana_fem": fem.median(),
            "mediana_masc": masc.median(),
            "U": u,
            "p_value": p,
            # positivo = feminino maior
            "rank_biserial": 2 * u / (len(fem) * len(masc)) - 1,
        })

gender_mw = pd.DataFrame(gender_mw_rows)
gender_mw["p_fdr"] = false_discovery_control(gender_mw["p_value"])
gender_mw

,condition,metric,n_fem,n_masc,mediana_fem,mediana_masc,U,p_value,rank_biserial,p_fdr
0,EO,Grau Médio,12,42,66020.456821,62984.377358,287.0,0.472871,0.138889,0.749855
1,EO,Coeficiente de Clustering Médio,12,42,0.548933,0.510840,294.0,0.387885,0.166667,0.749855
2,EO,Centralidade de Intermediação Médio,12,42,0.006648,0.007241,220.0,0.512212,-0.126984,0.749855
3,EC,Grau Médio,12,42,68099.951923,64261.804790,276.0,0.624879,0.095238,0.749855
4,EC,Coeficiente de Clustering Médio,12,42,0.579401,0.539627,292.0,0.411164,0.158730,0.749855
5,EC,Centralidade de Intermediação Médio,12,42,0.007210,0.007141,249.0,0.958516,-0.011905,0.958516


### Interação idade × condição

Diferença EO − EC de cada participante, comparada entre idosos e jovens. Só com essa diferença significativa dá para dizer que o efeito da idade muda entre olhos abertos e fechados.

In [ ]:
interaction_tests = pd.DataFrame(interaction_rows)
interaction_tests["p_fdr"] = false_discovery_control(interaction_tests["p_value"])
interaction_tests.round(4)

,metric,spearman_r,p_value
0,degree_mean,-0.425403,0.000004
1,clustering_mean,-0.422915,0.000005
2,betweenness_mean,-0.253127,0.008214


: 

A diferença EO − EC depende da idade em todas as métricas (p_FDR ≤ 0,019). Ao abrir os olhos, o grau e o agrupamento caem mais nos idosos (mediana EO − EC do grau: −0,039 vs −0,010 nos jovens), a variabilidade do grau cai nos idosos e fica estável nos jovens, e a intermediação sobe nos idosos. Por isso as diferenças entre grupos etários são maiores em EO do que em EC. Mesmo em indivíduos saudáveis, o envelhecimento aparece associado a redes instantâneas mais esparsas e menos agrupadas, com uma resposta mais forte à abertura dos olhos.